<a href="https://colab.research.google.com/github/OliverBrenningmeyer/tourplanning_modules/blob/main/Kemmler_Export_File_With_TrackingLinks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Skript

In [33]:
import requests
import json
import datetime
import re
import csv
import os
from google.colab import files
import pandas as pd
import io

def write_table_to_csv(table_data, filename="export_tourdaten_sendungsverfolgung.csv"):
    """
    Writes the table data to a CSV file and triggers a download.
    """
    if not table_data:
        print("No data to display.")
        return

    df = pd.DataFrame(table_data)
    df.to_csv(filename, index=False, sep=';', encoding='utf-8')  # Write directly to file
    files.download(filename)  # Trigger the download

def extract_auftrags_nr(extra_info_str):
    """
    Extracts the first ID-like value from the extraInfo string.
    Searches for patterns like [ABC12345], ABC12345, or a sequence of digits.
    """
    if not extra_info_str:
        return ""

    # Pattern 1: Alphanumeric inside brackets, e.g., [KOM1492057] or [KRA6050585]
    match = re.search(r'\[([a-zA-Z0-9_.-]+)\]', extra_info_str)
    if match:
        return match.group(1)

    # Pattern 2: Alphanumeric not necessarily in brackets, e.g. KOM1492057
    # Prioritize those starting with common prefixes if available or longer sequences
    match = re.search(r'\b([a-zA-Z]{2,4}\d{5,})\b', extra_info_str) # e.g., KOM12345, KRA12345
    if match:
        return match.group(1)

    # Pattern 3: A sequence of 7 or more digits, e.g., 6000057859
    match = re.search(r'\b(\d{7,})\b', extra_info_str)
    if match:
        return match.group(1)

    # Fallback: if no specific pattern, take the first significant looking part if any.
    # This is a simple fallback and might need refinement based on more extraInfo examples.
    parts = extra_info_str.split(" - ")
    if parts and parts[0].strip():
         # Check if it looks like an ID (e.g., contains numbers)
        if re.search(r'\d', parts[0]):
            return parts[0].strip()

    return ""


def create_table_from_json(json_data_str):
    """
    Processes the JSON data and returns a list of dictionaries,
    where each dictionary represents a row in the table.
    """
    try:
        data = json.loads(json_data_str)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        return []

    table_data = []
    bex_order_id = data.get("displayName", "N/A")

    for stop in data.get("stops", []):
        # Lieferdatum (Logged Arrival Time)
        time_unix = stop.get("timeTo")
        if time_unix:
            # Convert Unix timestamp to YYYY-MM-DD
            lieferdatum = datetime.datetime.fromtimestamp(time_unix).strftime('%Y-%m-%d')
        else:
            lieferdatum = "N/A"

        # Auftr.-Nr. (from extraInfo)
        address_info = stop.get("address", {})
        extra_info = address_info.get("extraInfo", "")
        auftrags_nr = extract_auftrags_nr(extra_info)

        # Kunde/Baustelle
        kunde_baustelle = address_info.get("companyName", "N/A")

        # Ort
        ort = address_info.get("city", "N/A")

        # Live Tracking Link
        live_tracking_key = stop.get("liveTrackingKey", "")
        live_tracking_link = ""
        if live_tracking_key:
            live_tracking_link = f"https://mission.orbit.technology/public/livetracking/{live_tracking_key}"
        else:
            live_tracking_link = "N/A"

        row = {
            "Niederlassung": "n/a",
            "Lieferdatum": lieferdatum,
            "Auftr.-Nr.": auftrags_nr,
            "Ort": ort,
            "Kennzeichen": "n/a",
            "Kunde/Baustelle": kunde_baustelle,
            "Fahrer": "n/a",
            "Sendungsnummer:": bex_order_id,
            "Link zur Sendungsverfolgung": live_tracking_link
        }
        table_data.append(row)

    return table_data

In [34]:
# --- Main execution ---
if __name__ == "__main__":
    order_codes_str = input("Enter the order codes (comma-separated, e.g., C0CCX,VWZGP): ")
    order_codes = [code.strip() for code in order_codes_str.split(',')]

    all_table_data = []
    for order_code in order_codes:
        url = f"https://api.orbit.do/v5/tours/{order_code}"
        headers = {
            "accept": "application/json",
            "X-API-KEY": "49CIZFu134XZ3g1ZW1bdEQhP9AfuIodJ0Rw4svq8"
        }
        response = requests.get(url, headers=headers)

        if response.status_code == 200:
            json_data_str = response.text
            table_data = create_table_from_json(json_data_str)
            all_table_data.extend(table_data)

        else:
            print(f"Error fetching data for {order_code}. Status code: {response.status_code}")

    df = pd.DataFrame(all_table_data)
    display(df)
    write_table_to_csv(all_table_data)

Enter the order codes (comma-separated, e.g., C0CCX,VWZGP): C0CCX,VWZGP


,Niederlassung,Lieferdatum,Auftr.-Nr.,Ort,Kennzeichen,Kunde/Baustelle,Fahrer,Sendungsnummer:,Link zur Sendungsverfolgung
0,n/a,2025-05-08,6000058130,Tübingen,n/a,TÜB Kemmler Baustoffe GmbH,n/a,KTDDN2LNX-PLN,https://mission.orbit.technology/public/livetr...
1,n/a,2025-05-08,,Tübingen,n/a,Horn Hartstoffe GmbH,n/a,KTDDN2LNX-PLN,https://mission.orbit.technology/public/livetr...
2,n/a,2025-05-08,,Tübingen,n/a,Aicher Neubau Zufahrt nur,n/a,KTDDN2LNX-PLN,https://mission.orbit.technology/public/livetr...
3,n/a,2025-05-08,6000058161,Nürtingen,n/a,NÜR Kemmler Baustoffe GmbH,n/a,KTDDN2LNX-PLN,https://mission.orbit.technology/public/livetr...
4,n/a,2025-05-08,6000058161,Deizisau,n/a,Lager Fa. Zweigle unbedin,n/a,KTDDN2LNX-PLN,https://mission.orbit.technology/public/livetr...
5,n/a,2025-05-08,KRA6050622,Nürtingen,n/a,NÜR Kemmler Baustoffe Gmb,n/a,KTDDN2LNX-PLN,https://mission.orbit.technology/public/livetr...
6,n/a,2025-05-08,KLS17046823,Holzmaden,n/a,- Lager -,n/a,KTDDN2LNX-PLN,https://mission.orbit.technology/public/livetr...
7,n/a,2025-05-08,KLS17046827,Weilheim an der Teck,n/a,Ulmer Thomas,n/a,KTDDN2LNX-PLN,https://mission.orbit.technology/public/livetr...
8,n/a,2025-05-08,KLS17046817,Kirchheim unter Teck,n/a,--LAGER-- Fa. Lang,n/a,KTDDN2LNX-PLN,https://mission.orbit.technology/public/livetr...
9,n/a,2025-05-08,,Bad Urach,n/a,Lagerplatz Holzbau Werner,n/a,KTDDN2LNX-PLN,https://mission.orbit.technology/public/livetr...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>